# Airbnb Paris — Prédiction du prix par nuitée

**Notebook principal à rendre**

## Plan
1. Chargement & nettoyage des données
2. EDA (Exploration)
3. Feature engineering
4. Pipeline (ColumnTransformer)
5. Modélisation & comparaison (CV 5-fold)
6. Analyse des résidus
7. Export du modèle final

In [111]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn — Pipeline anti-fuite
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# sklearn — modèles
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# sklearn — métriques
from sklearn.metrics import mean_absolute_error, r2_score

# Export (projet final)
import json
import joblib

# Setup
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print("Setup OK · Versions :")
import sklearn; print(f"  scikit-learn : {sklearn.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")

Setup OK · Versions :
  scikit-learn : 1.5.1
  pandas       : 2.2.2
  numpy        : 1.26.4


## 1. Chargement & Premiers regards

In [116]:
df = pd.read_csv("../data/listings.csv")
print(f"✓ Chargé depuis CSV : {df.shape}")

print(f"\nTypes :")
print(df.dtypes.value_counts())

print(f"\nNaN :")
print(df.isnull().sum())

print(f"\nTarget (price) :")
print(df["price"].describe())

print(f"\nColumns :")
print(df.columns.tolist())


✓ Chargé depuis CSV : (84055, 79)

Types :
object     35
int64      24
float64    20
Name: count, dtype: int64

NaN :
id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                              0
                                                ...  
calculated_host_listings_count                      0
calculated_host_listings_count_entire_homes         0
calculated_host_listings_count_private_rooms        0
calculated_host_listings_count_shared_rooms         0
reviews_per_month                               19557
Length: 79, dtype: int64

Target (price) :
count      53963
unique      1634
top       $90.00
freq         801
Name: price, dtype: object

Columns :
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'hos

## 1.2 Nettoyage des colonnes inutiles

In [117]:
cols_to_drop = [
    # URLs / texte libre → pas prédictif
    'listing_url', 'picture_url', 'host_url', 'host_thumbnail_url', 'host_picture_url',
    'name', 'description', 'neighborhood_overview', 'host_about', 'host_location',

    # IDs / metadata scraping → pas prédictif
    'scrape_id', 'last_scraped', 'calendar_last_scraped', 'calendar_updated',
    'host_id', 'host_name', 'source', 'license',

    # Doublons / redondants
    'neighbourhood',              # doublon de neighbourhood_cleansed
    'neighbourhood_group_cleansed',  # trop agrégé (arrondissement > quartier)
    'host_neighbourhood',         # subjectif, peu renseigné
    'minimum_minimum_nights', 'maximum_minimum_nights',  # redondant avec minimum_nights
    'minimum_maximum_nights', 'maximum_maximum_nights',  # redondant avec maximum_nights
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',  # redondant
    'calculated_host_listings_count_entire_homes',  # redondant avec calculated_host_listings_count
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'number_of_reviews_ltm', 'number_of_reviews_l30d',  # redondant avec number_of_reviews

    # Dates → nécessiterait feature engineering, on skip pour l'instant
    'host_since', 'first_review', 'last_review',

    # Disponibilité → pas prédictif pour le prix
    'availability_30', 'availability_60', 'availability_90',
    'has_availability',
]

df = df.drop(columns=cols_to_drop)
print(f"Shape après drop : {df.shape}")
print(df.columns.tolist())
df.head(3)


Shape après drop : (84055, 40)
['id', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'calculated_host_listings_count', 'reviews_per_month']


,id,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,...,number_of_reviews,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,reviews_per_month
0,3109,within an hour,100%,100%,f,1.0,1.0,"['email', 'phone']",t,t,Observatoire,48.83191,2.31870,Entire rental unit,Entire home/apt,...,7,185,0,31,4185.0,5.00,5.00,5.00,5.00,5.00,5.00,5.00,f,1,0.08
1,5396,within an hour,100%,95%,f,2.0,4.0,"['email', 'phone']",t,t,Hôtel-de-Ville,48.85247,2.35835,Entire rental unit,Entire home/apt,...,452,69,52,255,29070.0,4.62,4.64,4.59,4.81,4.85,4.95,4.59,f,1,2.32
2,7397,within an hour,100%,67%,t,1.0,10.0,"['email', 'phone']",t,t,Hôtel-de-Ville,48.85909,2.35315,Entire rental unit,Entire home/apt,...,380,122,24,255,37995.0,4.73,4.80,4.45,4.92,4.89,4.94,4.74,f,1,2.20


## 1.3) Nettoyage de prix

In [118]:
df["price"] = df["price"].str.replace('[$,]', '', regex=True).astype(float)

n_before = len(df)
df = df[(df['price'] > 10) & (df['price'] <= 1000)]
print(f"Après filtre 10-1000€ : {df.shape}")
print(f"Prix médian : {df['price'].median()}€ | max : {df['price'].max()}€")

Après filtre 10-1000€ : (52532, 40)
Prix médian : 158.0€ | max : 1000.0€


## 2. EDA — Exploration des données

In [ ]:
# Distribution du prix (brut vs log)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["price"], bins=100, edgecolor="black")
axes[0].set_title("Distribution du prix (brut)")
axes[0].set_xlabel("Prix (€)")
axes[1].hist(np.log1p(df["price"]), bins=100, edgecolor="black", color="orange")
axes[1].set_title("Distribution du prix (log1p)")
axes[1].set_xlabel("log(1 + prix)")
plt.tight_layout()
plt.show()

In [ ]:
# Prix moyen par arrondissement
plt.figure(figsize=(14, 6))
price_by_neighbourhood = df.groupby("neighbourhood_cleansed")["price"].median().sort_values(ascending=False)
sns.barplot(x=price_by_neighbourhood.index, y=price_by_neighbourhood.values)
plt.xticks(rotation=45, ha="right")
plt.title("Prix médian par arrondissement")
plt.xlabel("Arrondissement")
plt.ylabel("Prix médian (€)")
plt.tight_layout()
plt.show()

In [ ]:
# Prix par type de logement
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="room_type", y="price")
plt.title("Distribution du prix par type de logement")
plt.xlabel("Type")
plt.ylabel("Prix (€)")
plt.ylim(0, 500)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation des features numériques avec le prix
numeric_cols = ["price", "accommodates", "bedrooms", "beds", "bathrooms",
                "number_of_reviews", "review_scores_rating"]
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
# Carte des valeurs manquantes (top 20 colonnes avec le plus de NaN)
nan_counts = df.isnull().sum().sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 6))
sns.barplot(x=nan_counts.values, y=nan_counts.index, color="salmon")
plt.title("Top 20 colonnes avec le plus de valeurs manquantes")
plt.xlabel("Nombre de NaN")
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
import ast

# Parsing amenities
df["amenities_list"] = df["amenities"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Créer au moins 5 features booléennes à partir des amenities
selected_amenities = ["Wifi", "TV", "Elevator", "Washer", "Dryer", "Air conditioning", "Balcony"]

for amenity in selected_amenities:
    col = "has_" + amenity.lower().replace(" ", "_").replace("/", "_")
    df[col] = df["amenities_list"].apply(lambda x, a=amenity: int(a in x))

In [ ]:
# Sélection des features
numeric_features = ["accommodates", "bedrooms", "beds", "bathrooms"]
categorical_features = ["neighbourhood_cleansed", "room_type", "property_type"]
amenity_features = [c for c in df.columns if c.startswith("has_")]

all_features = numeric_features + categorical_features + amenity_features

# Nettoyage NaN
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())
df[categorical_features] = df[categorical_features].fillna("Unknown")

X = df[all_features]
y = df["price"]

print(f"X: {X.shape}, y: {y.shape}")

## 4. Pipeline (ColumnTransformer)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features + amenity_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

## 5. Modélisation & Comparaison (CV 5-fold)

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    
    # TransformedTargetRegressor avec log1p/expm1
    ttr = TransformedTargetRegressor(
        regressor=pipe, func=np.log1p, inverse_func=np.expm1
    )
    
    scoring = {"r2": "r2", "mae": "neg_mean_absolute_error"}
    cv_results = cross_validate(ttr, X, y, cv=5, scoring=scoring)
    
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    mae_mean = -cv_results["test_mae"].mean()
    mae_std = cv_results["test_mae"].std()
    
    results[name] = {"r2_mean": r2_mean, "r2_std": r2_std, "mae_mean": mae_mean, "mae_std": mae_std}
    print(f"{name:20s} | R² = {r2_mean:.4f} ± {r2_std:.4f} | MAE = {mae_mean:.2f}€ ± {mae_std:.2f}€")

In [ ]:
# Tableau récap
results_df = pd.DataFrame(results).T
results_df.sort_values("r2_mean", ascending=False)

## 6. Analyse des résidus (meilleur modèle)

In [ ]:
# Entraîner le meilleur modèle sur train/test split
best_name = results_df["r2_mean"].idxmax()
print(f"Meilleur modèle: {best_name}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_pipe = Pipeline([("preprocessor", preprocessor), ("model", models[best_name])])
best_ttr = TransformedTargetRegressor(regressor=best_pipe, func=np.log1p, inverse_func=np.expm1)
best_ttr.fit(X_train, y_train)

y_pred = best_ttr.predict(X_test)
residuals = y_test - y_pred

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Prédiction (€)")
axes[0].set_ylabel("Résidu (€)")
axes[0].set_title("Résidus vs Prédictions")

axes[1].hist(residuals, bins=80)
axes[1].set_title("Distribution des résidus")

axes[2].scatter(y_test, y_pred, alpha=0.3, s=5)
axes[2].plot([0, y_test.max()], [0, y_test.max()], "r--")
axes[2].set_xlabel("Prix réel (€)")
axes[2].set_ylabel("Prix prédit (€)")
axes[2].set_title("Réel vs Prédit")

plt.tight_layout()
plt.show()

## 7. Export du modèle & métriques

In [ ]:
# Ré-entraîner sur tout le dataset
final_pipe = Pipeline([("preprocessor", preprocessor), ("model", models[best_name])])
final_ttr = TransformedTargetRegressor(regressor=final_pipe, func=np.log1p, inverse_func=np.expm1)
final_ttr.fit(X, y)

# Sauvegarder
joblib.dump(final_ttr, "../model.joblib")
print("Modèle sauvegardé → model.joblib")

# Métriques
metrics = {
    "best_model": best_name,
    "cv_r2_mean": results[best_name]["r2_mean"],
    "cv_r2_std": results[best_name]["r2_std"],
    "cv_mae_mean": results[best_name]["mae_mean"],
    "cv_mae_std": results[best_name]["mae_std"],
}

with open("../metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Métriques sauvegardées → metrics.json")
metrics